# 01 - Initialize SageMaker Feature Store & Ingestion

Setup Feature Groups for 30-day readmission prediction.

In [1]:
%pip install "sagemaker<3.0.0" "prettytable>=3.9" --upgrade -qqqq

Note: you may need to restart the kernel to use updated packages.


In [2]:
%env SAGEMAKER_SUPPRESS_V2_WARNING=1

env: SAGEMAKER_SUPPRESS_V2_WARNING=1


In [3]:
import os
from time import gmtime, sleep, strftime, time
import boto3
import numpy as np
import pandas as pd
from sagemaker import get_execution_role
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.session import Session

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [4]:
region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(
    service_name="sagemaker",
    region_name=region
)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)


role = get_execution_role()
default_s3_bucket_name = feature_store_session.default_bucket()
prefix = "carepath-analytics/feature-store"

athena_s3_staging_dir = f"s3://{default_s3_bucket_name}/athena/staging"

print(f"AWS Region: {region}")
print(f"Execution Role ARN: {role}")
print(f"Feature Store S3 Prefix: s3://{default_s3_bucket_name}/{prefix}")
print(f"Athena Connection S3 Staging Dir: {athena_s3_staging_dir}")

AWS Region: us-east-1
Execution Role ARN: arn:aws:iam::769592409248:role/LabRole
Feature Store S3 Prefix: s3://sagemaker-us-east-1-769592409248/carepath-analytics/feature-store
Athena Connection S3 Staging Dir: s3://sagemaker-us-east-1-769592409248/athena/staging


# Load Raw Diabetes Dataset

In [5]:
data_path = "../../data/diabetic_data.csv"
raw_df = pd.read_csv(data_path)
print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head(3)

Raw dataset shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


# Feature Engineering & Clinical Preprocessing
1. **Record Identifier (`primary_key`):** `encounter_id`
2. **Event Timestamp (`event_time`):** Unix epoch timestamp
3. **Target Transformation:** Convert 3-level target `readmitted` categorical into binary `readmitted_30d` (1 if `<30` else 0).
4. **ICD-9 Grouping:** Convert diagnosis codes (`diag_1`, `diag_2`, `diag_3`) into clinical categories.
5. **Null Handling:** Standardize null/NaN/? values into "Unknown"

In [6]:
df = raw_df.copy()

# Null handling
df.replace("?", "Unknown", inplace=True)
df.fillna("Unknown", inplace=True)

# Target transformation
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)


# ICD-9 Grouping
def categorize_icd9(code):
    if code is None:
        return "Unknown"

    code_str = str(code).strip().upper()
    if not code_str or code_str in ["UNKNOWN", "NAN", "NONE"]:
        return "Unknown"
    if code_str.startswith(("V", "E")):
        return "Other"
    try:
        val = int(code_str.split(".")[0][:3])
    except ValueError:
        return "Other"

    if (390 <= val <= 459) or val == 785:
        return "Circulatory"
    elif (460 <= val <= 519) or val == 786:
        return "Respiratory"
    elif (520 <= val <= 579) or val == 787:
        return "Digestive"
    elif val == 250:
        return "Diabetes"
    elif (800 <= val <= 999):
        return "Injury"
    elif (710 <= val <= 739):
        return "Musculoskeletal"
    elif (580 <= val <= 629) or val == 788:
        return "Genitourinary"
    elif (140 <= val <= 239):
        return "Neoplasms"
    else:
        return "Other"


df["diag_1_group"] = df["diag_1"].apply(categorize_icd9)
df["diag_2_group"] = df["diag_2"].apply(categorize_icd9)
df["diag_3_group"] = df["diag_3"].apply(categorize_icd9)

# Set event_time
df["event_time"] = float(time())

# Feature selection for FeatureGroup
selected_feature_columns = [
    "encounter_id",
    "patient_nbr",
    "event_time",
    "race",
    "gender",
    "age",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "diag_1_group",
    "diag_2_group",
    "diag_3_group",
    "max_glu_serum",
    "A1Cresult",
    "change",
    "diabetesMed",
    "readmitted_30d",
]

feature_df = df[selected_feature_columns].copy()

# Set numerical features to int
int_cols = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "readmitted_30d",
]
for col in int_cols:
    feature_df[col] = feature_df[col].astype(int)

# Set categorical features to string
str_cols = [
    "encounter_id",
    "patient_nbr",
    "race",
    "gender",
    "age",
    "diag_1_group",
    "diag_2_group",
    "diag_3_group",
    "max_glu_serum",
    "A1Cresult",
    "change",
    "diabetesMed",
]
for col in str_cols:
    feature_df[col] = feature_df[col].astype(str)

print(f"Shape: {feature_df.shape}")
print(f"Readmitted (<30d) positive class count: {feature_df['readmitted_30d'].sum()}")
feature_df.head(3)

Shape: (101766, 25)
Readmitted (<30d) positive class count: 11357


,encounter_id,patient_nbr,event_time,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,number_inpatient,number_diagnoses,diag_1_group,diag_2_group,diag_3_group,max_glu_serum,A1Cresult,change,diabetesMed,readmitted_30d
0,2278392,8222157,1.790317e+09,Caucasian,Female,[0-10),6,25,1,1,...,0,1,Diabetes,Unknown,Unknown,Unknown,Unknown,No,No,0
1,149190,55629189,1.790317e+09,Caucasian,Female,[10-20),1,1,7,3,...,0,9,Other,Diabetes,Other,Unknown,Unknown,Ch,Yes,0
2,64410,86047875,1.790317e+09,AfricanAmerican,Female,[20-30),1,1,7,2,...,1,6,Other,Diabetes,Other,Unknown,Unknown,No,Yes,0


# Create SageMaker Feature Group

In [7]:
feature_group_name = "diabetes-feature-group-" + strftime("%d-%H-%M-%S", gmtime())

feature_group = FeatureGroup(
    name=feature_group_name,
    sagemaker_session=feature_store_session,
)

# Infer feature schema from DataFrame dtypes
feature_group.load_feature_definitions(data_frame=feature_df)

print(f"Creating FeatureGroup: {feature_group_name}...")
feature_group.create(
    s3_uri=f"s3://{default_s3_bucket_name}/{prefix}",
    record_identifier_name="encounter_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True,
)


# Helper function to poll status
def check_feature_group_status(fg):
    status = fg.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group to be Created...")
        sleep(5)
        status = fg.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Feature Group creation failed with status: {status}")
    print(f"FeatureGroup '{fg.name}' successfully created.")


check_feature_group_status(feature_group)

Creating FeatureGroup: diabetes-feature-group-25-06-12-59...
Waiting for Feature Group to be Created...
Waiting for Feature Group to be Created...
Waiting for Feature Group to be Created...
Waiting for Feature Group to be Created...
FeatureGroup 'diabetes-feature-group-25-06-12-59' successfully created.


In [8]:
feature_group.describe()

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:769592409248:feature-group/diabetes-feature-group-25-06-12-59',
 'FeatureGroupName': 'diabetes-feature-group-25-06-12-59',
 'RecordIdentifierFeatureName': 'encounter_id',
 'EventTimeFeatureName': 'event_time',
 'FeatureDefinitions': [{'FeatureName': 'encounter_id',
   'FeatureType': 'String'},
  {'FeatureName': 'patient_nbr', 'FeatureType': 'String'},
  {'FeatureName': 'event_time', 'FeatureType': 'Fractional'},
  {'FeatureName': 'race', 'FeatureType': 'String'},
  {'FeatureName': 'gender', 'FeatureType': 'String'},
  {'FeatureName': 'age', 'FeatureType': 'String'},
  {'FeatureName': 'admission_type_id', 'FeatureType': 'Integral'},
  {'FeatureName': 'discharge_disposition_id', 'FeatureType': 'Integral'},
  {'FeatureName': 'admission_source_id', 'FeatureType': 'Integral'},
  {'FeatureName': 'time_in_hospital', 'FeatureType': 'Integral'},
  {'FeatureName': 'num_lab_procedures', 'FeatureType': 'Integral'},
  {'FeatureName': 'num_procedures'

# Ingest Patient Records into Feature Group

In [9]:
# Set sample_size = len(feature_df) when executing the full production pipeline
# For testing, use a small sample size to reduce waiting time here
sample_size = 5000
ingest_df = feature_df.head(sample_size).copy()

print(f"Ingesting {len(ingest_df)} records into {feature_group_name}...")
feature_group.ingest(
    data_frame=ingest_df,
    max_workers=max(1, os.cpu_count() - 1),
    wait=True,
)
print("Ingestion complete!")

Ingesting 5000 records into diabetes-feature-group-25-06-12-59...
Ingestion complete!


# Validate Online Store Retrieval
Execute lookups via `batch_get_record` to verify that feature group is operational.

In [10]:
def get_feature_value(record, feature_name):
    matches = [r["ValueAsString"] for r in record if r["FeatureName"] == feature_name]
    return matches[0] if matches else None


# Select 3 sample encounter IDs from the ingested batch
sample_encounters = ingest_df["encounter_id"].head(3).tolist()

response = featurestore_runtime.batch_get_record(
    Identifiers=[
        {
            "FeatureGroupName": feature_group_name,
            "RecordIdentifiersValueAsString": sample_encounters,
        }
    ]
)

retrieved_records = response.get("Records", [])
query_results = pd.DataFrame(
    [
        [get_feature_value(r["Record"], col) for col in selected_feature_columns]
        for r in retrieved_records
    ],
    columns=selected_feature_columns,
)

print(f"Successfully queried {len(query_results)} records from Online Feature Store:")
query_results[["encounter_id", "patient_nbr", "age", "diag_1_group", "readmitted_30d"]]

Successfully queried 3 records from Online Feature Store:


,encounter_id,patient_nbr,age,diag_1_group,readmitted_30d
0,2278392,8222157,[0-10),Diabetes,0
1,149190,55629189,[10-20),Other,0
2,64410,86047875,[20-30),Other,0
